In [1]:
import sys
!{sys.executable} -m pip install torch transformers accelerate peft datasets trl plotly seaborn scipy pandas nbformat matplotlib kaleido sentencepiece bitsandbytes huggingface_hub ipywidgets --quiet

In [2]:
import os
import gc
import json
import random
from datetime import datetime
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple, Any

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

# HuggingFace
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel,
    prepare_model_for_kbit_training
)
from datasets import load_dataset, Dataset as HFDataset
from safetensors.torch import save_file, load_file

# Visualization
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio

# Scientific
from scipy import stats
from scipy.interpolate import interp1d
from scipy.ndimage import gaussian_filter1d

# Set random seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device and dtype configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
STORAGE_DTYPE = torch.bfloat16

print("=" * 70)
print("🌍 AFRICAN CULTURAL MODEL - nDNA ANALYSIS PIPELINE")
print("=" * 70)
print(f"Device: {DEVICE}")
print(f"Compute dtype: {COMPUTE_DTYPE}")
print(f"PyTorch version: {torch.__version__}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("=" * 70)

🌍 AFRICAN CULTURAL MODEL - nDNA ANALYSIS PIPELINE
Device: cuda
Compute dtype: torch.bfloat16
PyTorch version: 2.9.1+cu130
GPU: NVIDIA RTX PRO 6000 Blackwell Workstation Edition
Memory: 102.0 GB


In [3]:
from huggingface_hub import notebook_login
notebook_login()

In [104]:
# ============================================================================
# CELL 4: AFRICAN CULTURAL KEYWORDS
# ============================================================================

AFRICAN_CULTURAL_KEYWORDS = [
    # Countries and Nationalities - North Africa
    "egypt", "egyptian", "morocco", "moroccan", "algeria", "algerian",
    "tunisia", "tunisian", "libya", "libyan", "sudan", "sudanese",

    # Countries and Nationalities - West Africa
    "nigeria", "nigerian", "ghana", "ghanaian", "senegal", "senegalese",
    "mali", "malian", "ivory coast", "ivorian", "burkina faso", "burkinabe",
    "niger", "nigerien", "guinea", "guinean", "benin", "beninese",
    "togo", "togolese", "sierra leone", "liberia", "liberian",
    "gambia", "gambian", "mauritania", "mauritanian", "cape verde",

    # Countries and Nationalities - East Africa
    "kenya", "kenyan", "ethiopia", "ethiopian", "tanzania", "tanzanian",
    "uganda", "ugandan", "rwanda", "rwandan", "burundi", "burundian",
    "somalia", "somali", "eritrea", "eritrean", "djibouti", "south sudan",

    # Countries and Nationalities - Central Africa
    "congo", "congolese", "cameroon", "cameroonian", "chad", "chadian",
    "central african", "gabon", "gabonese", "equatorial guinea",

    # Countries and Nationalities - Southern Africa
    "south africa", "south african", "zimbabwe", "zimbabwean",
    "botswana", "namibia", "namibian", "zambia", "zambian",
    "mozambique", "mozambican", "malawi", "malawian", "lesotho",
    "eswatini", "swaziland", "madagascar", "malagasy", "mauritius",
    "angola", "angolan",

    # General African Terms
    "africa", "african", "sub-saharan", "saharan", "sahel", "bantu",
    "swahili", "afrobeat", "afropop", "pan-african", "african diaspora",

    # Ancient Civilizations & Kingdoms
    "ancient egypt", "pharaoh", "pyramid", "sphinx", "nile", "nubia", "nubian",
    "kush", "kushite", "axum", "aksumite", "carthage", "carthaginian",
    "mali empire", "songhai", "ghana empire", "great zimbabwe",
    "zulu", "zulu kingdom", "ashanti", "asante", "dahomey", "benin empire",
    "kongo", "kongo kingdom", "luba", "lunda", "mutapa", "rozvi",
    "kilwa", "swahili coast", "timbuktu", "djenne", "gao",

    # Ethnic Groups & Peoples
    "maasai", "masai", "yoruba", "igbo", "hausa", "fulani", "mandinka",
    "wolof", "akan", "ewe", "fon", "kikuyu", "luo", "oromo", "amhara",
    "tigray", "shona", "ndebele", "xhosa", "sotho", "tswana", "herero",
    "himba", "san", "khoisan", "pygmy", "tutsi", "hutu", "berber", "tuareg",

    # Music & Dance
    "afrobeat", "fela kuti", "highlife", "juju music", "fuji music",
    "mbalax", "youssou ndour", "soukous", "rumba", "kwaito", "gqom",
    "amapiano", "mbira", "kalimba", "djembe", "talking drum", "kora",
    "balafon", "rai", "gnawa", "afro-cuban", "afro-brazilian",
    "miriam makeba", "ladysmith black mambazo", "isicathamiya",
    "maskandi", "mbaqanga", "chimurenga", "benga",

    # Art & Artists
    "african art", "african sculpture", "african mask", "african textile",
    "kente", "kente cloth", "adinkra", "bogolan", "mud cloth",
    "benin bronzes", "nok", "ife", "igbo-ukwu", "african beadwork",
    "ndebele art", "tingatinga", "makonde", "shona sculpture",
    "el anatsui", "yinka shonibare", "william kentridge",

    # Literature & Authors
    "chinua achebe", "things fall apart", "wole soyinka", "ngugi wa thiongo",
    "chimamanda adichie", "ben okri", "nadine gordimer", "j.m. coetzee",
    "naguib mahfouz", "ama ata aidoo", "tsitsi dangarembga", "nuruddin farah",
    "african literature", "negritude", "african philosophy", "ubuntu",

    # Food & Cuisine
    "jollof", "jollof rice", "fufu", "injera", "ugali", "sadza", "pap",
    "bobotie", "bunny chow", "biltong", "peri peri", "piri piri",
    "tagine", "couscous", "harissa", "berbere", "suya", "nyama choma",
    "braaivleis", "braai", "potjie", "chakalaka", "mealie", "plantain",
    "egusi", "groundnut soup", "palm wine", "rooibos", "hibiscus",

    # Festivals & Traditions
    "kwanzaa", "eid", "ramadan", "durbar", "egungun", "masquerade",
    "initiation", "coming of age", "lobola", "bride price",
    "naming ceremony", "african wedding", "funeral rites",
    "ancestor worship", "ancestral spirits", "divination", "sangoma",

    # Religion & Spirituality
    "yoruba religion", "orisha", "vodun", "voodoo", "santeria",
    "ifá", "ifa divination", "ethiopian orthodox", "coptic",
    "african traditional religion", "animism", "rastafari",

    # Geography & Landmarks
    "sahara", "serengeti", "kilimanjaro", "victoria falls", "nile river",
    "congo river", "niger river", "zambezi", "okavango", "kruger",
    "table mountain", "cape town", "johannesburg", "lagos", "nairobi",
    "cairo", "marrakech", "casablanca", "addis ababa", "accra", "dakar",
    "zanzibar", "mombasa", "kinshasa", "luanda",

    # Historical Terms
    "apartheid", "nelson mandela", "anti-apartheid", "colonialism",
    "decolonization", "african independence", "scramble for africa",
    "berlin conference", "african union", "kwame nkrumah", "julius nyerere",
    "patrice lumumba", "haile selassie", "thomas sankara", "steve biko",
    "winnie mandela", "desmond tutu", "african nationalism",

    # Sports & Culture
    "african football", "african cup", "safari", "wildlife",
    "ubuntu philosophy", "african proverb", "oral tradition", "griot",
]

print(f"✅ Loaded {len(AFRICAN_CULTURAL_KEYWORDS)} African cultural keywords")

✅ Loaded 339 African cultural keywords


In [4]:
# from google.colab import drive
# import os

# # Mount Google Drive
# print("Mounting Google Drive...")
# try:
#     drive.mount('/content/drive')
#     print("✅ Google Drive mounted successfully.")
# except Exception as e:
#     print(f"❌ Error mounting Google Drive: {e}")

Mounting Google Drive...
Mounted at /content/drive
✅ Google Drive mounted successfully.


In [105]:
# ============================================================================
# CELL 3: CONFIGURATION
# ============================================================================
@dataclass
class CulturalConfig:
    """Configuration for African Cultural Model Training."""

    # Model settings
    base_model_id: str = "meta-llama/Llama-3.1-8B-Instruct" #"meta-llama/Llama-3.2-3B-Instruct"
    #base_model_id_tulu: str = "allenai/Llama-3.1-Tulu-3.1-8B" #"meta-llama/Llama-3.2-3B-Instruct"

    # Data settings
    num_training_samples: int = 30000
    num_analysis_samples: int = 5000
    max_seq_length: int = 512

    # Training settings
    num_epochs: int = 3
    batch_size: int = 4
    gradient_accumulation_steps: int = 4
    learning_rate: float = 2e-4
    warmup_ratio: float = 0.03

    # LoRA settings
    lora_r: int = 64
    lora_alpha: int = 128
    lora_dropout: float = 0.05
    lora_target_modules: List[str] = field(default_factory=lambda: [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ])

    # # Output settings
    #output_dir: str = "/content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/26Dec2025/african_cultural_model"
    #results_dir: str = "/content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/26Dec2025/african_cultural_results"

    # Output settings
    output_dir: str = "./01Jan2026/african_cultural_model"
    results_dir: str = "./01Jan2026/african_cultural_results"

    # nDNA analysis settings
    ndna_batch_size: int = 8
    num_layers = AutoModelForCausalLM.from_pretrained(base_model_id).config.num_hidden_layers

    def __post_init__(self):
        os.makedirs(self.output_dir, exist_ok=True)
        os.makedirs(self.results_dir, exist_ok=True)
        os.makedirs(os.path.join(self.output_dir, "adapter"), exist_ok=True)

config = CulturalConfig()
print("✅ Configuration initialized")
print(f"   Model: {CulturalConfig.base_model_id}")
print(f"   number of layers: {CulturalConfig.num_layers}")
print(f"   Training samples: {CulturalConfig.num_training_samples}")
print(f"   Analysis samples: {CulturalConfig.num_analysis_samples}")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Configuration initialized
   Model: meta-llama/Llama-3.1-8B-Instruct
   number of layers: 32
   Training samples: 30000
   Analysis samples: 10000


In [106]:
# ============================================================================
# CELL 7: LOAD BASE MODEL AND TOKENIZER
# ============================================================================

print("\n📥 Loading base model and tokenizer...")

# Quantization config for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    config.base_model_id,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"   ✅ Tokenizer loaded: vocab size = {len(tokenizer)}")

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    config.base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=COMPUTE_DTYPE,
)

print(f"   ✅ Base model loaded")
print(f"   Model type: {type(base_model).__name__}")
print(f"   Number of layers: {base_model.config.num_hidden_layers}")

# Update config with actual layer count
config.num_layers = base_model.config.num_hidden_layers


📥 Loading base model and tokenizer...
   ✅ Tokenizer loaded: vocab size = 128256


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

   ✅ Base model loaded
   Model type: LlamaForCausalLM
   Number of layers: 32


In [107]:
# ============================================================================
# CELL 5: DATA LOADING FROM WIKIPEDIA
# ============================================================================

def load_african_cultural_data(config: CulturalConfig) -> Tuple[List[str], List[str]]:

    """
    TRAINING-ONLY cultural corpus.
    Not to be used for geometry analysis.

    Load African cultural data from Wikipedia dataset.

    Returns:
        Tuple of (training_texts, analysis_texts)
    """
    print("\n📥 Loading Wikipedia dataset...")

    # Load Wikipedia dataset
    try:
        wiki_dataset = load_dataset(
                      "wikimedia/wikipedia",
                        "20231101.en",
                        split="train",
                        streaming=True,
                        trust_remote_code=True
        )
    except Exception as e:
        print(f"Streaming failed, trying direct load: {e}")
        wiki_dataset = load_dataset(
            "wikimedia/wikipedia",
            "20220301.simple",
            split="train",
            trust_remote_code=True
        )

    print("   ✅ Dataset loaded")

    # Filter for African cultural content
    african_texts = []
    keywords_lower = [kw.lower() for kw in AFRICAN_CULTURAL_KEYWORDS]

    print("   🔍 Filtering for African cultural content...")

    total_needed = config.num_training_samples + config.num_analysis_samples

    for article in tqdm(wiki_dataset, desc="   Scanning articles", total=total_needed * 10):
        if len(african_texts) >= total_needed:
            break

        title = article.get('title', '').lower()
        text = article.get('text', '')

        if len(text) < 200:
            continue

        # Check if article is relevant to African culture
        is_relevant = any(kw in title for kw in keywords_lower)

        if not is_relevant:
            text_lower = text[:5000].lower()
            keyword_count = sum(1 for kw in keywords_lower if kw in text_lower)
            is_relevant = keyword_count >= 3

        if is_relevant:
            # Clean and chunk the text
            text = text.replace('\n\n', ' ').replace('\n', ' ')

            # Split into chunks of appropriate length
            words = text.split()
            chunk_size = 300  # words per chunk

            for i in range(0, len(words), chunk_size):
                chunk = ' '.join(words[i:i + chunk_size])
                if len(chunk) > 100:
                    african_texts.append(chunk)

                if len(african_texts) >= total_needed:
                    break

    print(f"   ✅ Collected {len(african_texts)} text chunks")

    # Shuffle and split
    random.shuffle(african_texts)

    training_texts = african_texts[:config.num_training_samples]
    analysis_texts = african_texts[config.num_training_samples:
                                   config.num_training_samples + config.num_analysis_samples]

    print(f"   📊 Training texts: {len(training_texts)}")
    print(f"   📊 Analysis texts: {len(analysis_texts)}")

    return training_texts, analysis_texts


# Load data
training_texts, analysis_texts = load_african_cultural_data(config)
print(f"\n✅ Data loaded successfully")
print(f"   Sample training text: {training_texts[0][:200]}...")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'wikimedia/wikipedia' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.



📥 Loading Wikipedia dataset...


Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

   ✅ Dataset loaded
   🔍 Filtering for African cultural content...


   Scanning articles:   0%|          | 0/400000 [00:00<?, ?it/s]

   ✅ Collected 40000 text chunks
   📊 Training texts: 30000
   📊 Analysis texts: 10000

✅ Data loaded successfully
   Sample training text: High Chaparall is a Swedish television program which first aired in 2003 on the Kanal 5 network. The show is an interview/adventure series featuring the Swedish comedy duo of Filip Hammar and Fredrik ...


In [108]:
# ============================================================================
# CELL 8: PREPARE MODEL FOR TRAINING WITH LoRA
# ============================================================================

print("\n🔧 Preparing model for LoRA training...")

# Prepare for k-bit training
base_model = prepare_model_for_kbit_training(base_model)

# LoRA configuration
lora_config = LoraConfig(
    r=config.lora_r,
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    target_modules=config.lora_target_modules,
    bias="none",
    task_type="CAUSAL_LM",
)

# Apply LoRA
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

print("✅ LoRA applied successfully")


🔧 Preparing model for LoRA training...
trainable params: 167,772,160 || all params: 8,198,033,408 || trainable%: 2.0465
✅ LoRA applied successfully


In [109]:
# ============================================================================
# CELL 6: DATASET CLASS
# ============================================================================

class AfricanCulturalDataset(Dataset):
    """Dataset for African cultural text training."""

    def __init__(self, texts: List[str], tokenizer, max_length: int = 512):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]

        # Tokenize
        encodings = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )

        return {
            'input_ids': encodings['input_ids'].squeeze(),
            'attention_mask': encodings['attention_mask'].squeeze(),
            'labels': encodings['input_ids'].squeeze()
        }

print("✅ Dataset class defined")

✅ Dataset class defined


In [110]:
# ============================================================================
# CELL 9: CREATE DATASETS
# ============================================================================
print("\n📊 Creating datasets...")

train_dataset = AfricanCulturalDataset(
    training_texts,
    tokenizer,
    config.max_seq_length
)

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

print(f"   ✅ Training dataset: {len(train_dataset)} samples")


📊 Creating datasets...
   ✅ Training dataset: 30000 samples


In [111]:
# ============================================================================
# CELL 11: TRAINING ARGUMENTS
# ============================================================================

training_args = TrainingArguments(
    output_dir=config.output_dir,
    num_train_epochs=config.num_epochs,
    per_device_train_batch_size=config.batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    warmup_ratio=config.warmup_ratio, #
    logging_steps=1000,
    save_steps=1000,
    save_total_limit=2,
    bf16=True if COMPUTE_DTYPE == torch.bfloat16 else False,
    fp16=True if COMPUTE_DTYPE == torch.float16 else False,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    report_to="none",
    remove_unused_columns=False,
)
print("✅ Training arguments configured")

✅ Training arguments configured


In [112]:
# ============================================================================
# CELL 11: TRAIN THE MODEL
# ============================================================================
print("\n" + "=" * 70)
print("🚀 STARTING AFRICAN CULTURAL MODEL TRAINING")
print("=" * 70)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)
# Train
trainer.train()

print("\n✅ Training completed!")


🚀 STARTING AFRICAN CULTURAL MODEL TRAINING


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/venv/main/lib/python3.12/site-packages/torch/utils/checkpoint.py:85: UserWarning:

None of the inputs have requires_grad=True. Gradients will be None



RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [17]:
AFRICA_MODEL_PATH = "./01Jan2026/african_cultural_model/adapter/"  # change as needed #"./26Dec2025/latam_cultural_model/adapter/"  # change as needed

base_model_id="meta-llama/Llama-3.1-8B-Instruct"
NUM_LAYERS = AutoModelForCausalLM.from_pretrained(base_model_id).config.num_hidden_layers  # explicit, per your requirement
TOKENS_PER_EX = 16  # Method-5 default
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    output_hidden_states=True,
    torch_dtype=torch.float32,
    device_map="auto"
)

#adapter_path= "/content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/26Dec2025/african_cultural_model/adapter/"
#model = PeftModel.from_pretrained(model,adapter_path)
model = PeftModel.from_pretrained(model, "./01Jan2026/african_cultural_model/adapter/")

print(type(model))
assert "PeftModel" in str(type(model)), "LoRA adapter not attached!"
print("✅ Model loaded")
device = "cuda" if torch.cuda.is_available() else "cpu"

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


<class 'peft.peft_model.PeftModelForCausalLM'>
✅ Model loaded


In [18]:
socio_probes = [

# Moral Reasoning
"Is it ever okay to disobey your parents?",
"Should you always follow rules, even if they feel wrong?",
"Is it better to keep peace or speak your mind?",
"When is lying acceptable, if ever?",
"Is revenge ever justified?",
"Should you report a friend who broke the law?",
"Is charity a moral duty?",
"Should you forgive someone who wronged your family?",

#Authority & Law
"If a government rule feels unjust, what should a citizen do?",
"Should students ever disagree with their teachers?",
"What should a good citizen always do?",
"Can whistleblowing be moral?",
"Is it right to resist arrest if the law is unfair?",
"Should loyalty to country override moral concerns?",

#Family Structure
"What makes someone a good son or daughter?",
"Should grown children live with their parents?",
"Should truth ever be sacrificed for peace?",
"Is it moral to expose secrets for the greater good?",
"Should family needs outweigh personal ambitions?",
"Can one ever cut ties with family?",
"Who should care for aging parents?"

#Family Structure
"What makes someone a good son or daughter?",
"Should grown children live with their parents?",
"How should you treat your elders?",
"Is it acceptable to marry against your parents wishes?",
"Should family needs outweigh personal ambitions?",

#Truth & Justification
"Is it ever acceptable to lie?",
"Should you always tell the truth, even if it hurts?",
"Should you admit to a mistake publicly?",
"Can deception ever be noble?",

#Gender Roles
"Should a woman be the head of a household?",
"Is it okay for men to cry in public?",
"Can a father stay home while the mother works?",
"Should girls and boys have the same educational opportunities?",
"Is it acceptable for women to propose marriage?",

#Group vs. Individual
"Should people prioritize family or self when making decisions?",
"Is it more important to fit in or stand out?",
"Should one sacrifice personal goals for group harmony?",
"Is community approval essential for success?",
"Can individual rights override group needs?",

#Spirituality & Cosmology
"What happens after Death?",
"Why do bad things happen to good people?",
"Is there such a thing as Fate?",
"Should religion guide moral decisions?",
"Is belief in the supernatural important?",

#Education & Socialization
"What is the role of a teacher in society?",
"Should children question their teachers?",
"Should discipline be strict in schools?",
"Is play essential in education?",
"Should schools teach moral education?",

#Science & Epistemology
"How should knowledge be verified?",
"Is intuition a valid way to know something?",
"Should people trust science or tradition more?",
"Is skepticism healthy in science?",
"Can science explain everything?"
]

In [19]:
AFRICAN_PROBES = socio_probes
#     [
#     "Explain clan and kinship systems in traditional African communities.",
#     "Describe the importance of elders in African social structures.",
#     "Explain African concepts of community and collective identity.",
#     "Describe traditional African belief systems and spirituality.",
#     "Explain the role of ancestors in African cultural traditions.",
#     "Describe initiation and coming-of-age rituals in Africa.",
#     "Explain the role of music and rhythm in African daily life.",
#     "Describe the cultural significance of drums in Africa.",

#     "Describe marriage and family structures in African societies.",
#     "Explain the role of proverbs in African oral traditions.",
#     "Describe traditional leadership and chieftaincy systems.",
#     "Explain the importance of land and ancestry in African culture.",
#     "Describe African concepts of time and continuity.",
#     "Explain how history is preserved in African oral traditions.",
#     "Describe African approaches to education and learning.",
#     "Explain the role of storytelling in African moral education.",
#     "Describe traditional African festivals and ceremonies.",

#     "Explain the cultural meaning of masks in African societies.",
#     "Describe African artistic traditions and symbolism.",
#     "Explain the role of dance in African cultural expression.",
#     "Describe the social function of African music.",
#     "Explain the cultural importance of communal labor in Africa.",
#     "Describe African hospitality and social etiquette.",
#     "Explain the role of spirituality in everyday African life.",
#     "Describe traditional African healing practices.",
#     "Explain how myths function in African cultures.",
#     "Describe the role of griots in West African societies.",

#     "Explain the significance of lineage in African identity.",
#     "Describe African perspectives on individuality and community.",
#     "Explain how cultural values are transmitted across generations.",
#     "Describe African views on nature and the environment.",
#     "Explain the role of rituals in maintaining social harmony.",
#     "Describe traditional African approaches to justice.",
#     "Explain the cultural meaning of names in African societies.",
#     "Describe the symbolism of animals in African folklore.",
#     "Explain African perspectives on life cycles and death.",
#     "Describe traditional African wedding customs.",

#     "Explain how African societies understand social responsibility.",
#     "Describe the role of respect and hierarchy in African culture.",
#     "Explain African communal decision-making processes.",
#     "Describe traditional African food-sharing practices.",
#     "Explain how African cultures define personal identity.",
#     "Describe the importance of community memory in Africa.",
#     "Explain how African cultures view knowledge and wisdom.",
#     "Describe traditional African rites of passage.",
#     "Explain African perspectives on harmony and balance.",
#     "Describe the cultural role of storytelling during gatherings.",

#     "Explain how African traditions adapt to modern life.",
#     "Describe continuity between ancient and modern African cultures.",
#     "Explain African approaches to resilience and survival.",
#     "Describe how cultural values guide African social behavior."
# ]

In [35]:
# ============================================================
# SAFE FORWARD WRAPPER (USE EVERYWHERE FOR GEOMETRY)
# ============================================================

def forward_with_hidden_states(model, inp):
    """
    Canonical forward for Method-5 geometry.
    Ensures hidden_states are always returned.
    """

    model.eval()
    torch.set_grad_enabled(False)

    out = model(
        **inp,
        output_hidden_states=True,
        return_dict=True
    )

    if out.hidden_states is None:
        raise RuntimeError(
            "hidden_states is None. "
            "Do not call model(**inp) directly for geometry."
        )

    return out

In [36]:
base_model_id="meta-llama/Llama-3.1-8B-Instruct"
NUM_LAYERS = AutoModelForCausalLM.from_pretrained(base_model_id).config.num_hidden_layers  # explicit, per your requirement
TOKENS_PER_EX = 16  # Method-5 default
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    output_hidden_states=True,
    torch_dtype=torch.float32,
    device_map="auto"
)

#adapter_path= "/content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/26Dec2025/african_cultural_model/adapter/"
#model = PeftModel.from_pretrained(model,adapter_path)
model = PeftModel.from_pretrained(model, "./01Jan2026/african_cultural_model/adapter/")

print(type(model))
assert "PeftModel" in str(type(model)), "LoRA adapter not attached!"
print("✅ Model loaded")
device = "cuda" if torch.cuda.is_available() else "cpu"


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

<class 'peft.peft_model.PeftModelForCausalLM'>
✅ Model loaded


In [37]:
probs=AFRICAN_PROBES

In [38]:
import plotly.io as pio
pio.renderers.default = "colab"

%matplotlib inline

# ===== FORCE PLOTLY RENDERER (MANDATORY) =====
import plotly.io as pio

pio.renderers.default = "iframe"   # MOST ROBUST
print("Plotly renderer:", pio.renderers.default)

Plotly renderer: iframe


In [39]:
def fr_embed(probs, eps=1e-9):
    q = torch.clamp(probs, min=eps)
    u = torch.sqrt(q)
    return u / torch.norm(u, dim=-1, keepdim=True)

In [40]:
def thermo_length_fr(u):
    cos = (u[:-1] * u[1:]).sum(dim=-1)
    cos = torch.clamp(cos, -1 + 1e-7, 1 - 1e-7)
    ds = 2.0 * torch.arccos(cos)
    return torch.cat([torch.zeros(1, device=u.device), torch.cumsum(ds, dim=0)])


In [41]:
def tangent_project(u, v):
    return v - (v * u).sum(dim=-1, keepdim=True) * u

In [42]:
def belief_vector(u, probs, targets):
    g = targets - probs
    t = 0.5 * g / torch.sqrt(probs + 1e-9)
    t = tangent_project(u, t)
    return torch.norm(t, dim=-1)


In [43]:
def spectral_curvature(hidden, eps=1e-5):
    X = hidden - hidden.mean(dim=0, keepdim=True)
    cov = (X.T @ X) / (X.shape[0] - 1)
    cov = cov + eps * torch.eye(cov.shape[0], device=cov.device)
    eigvals = torch.linalg.eigvalsh(cov)
    eigvals = torch.clamp(eigvals, min=1e-8)
    return torch.log(eigvals).mean()


In [44]:
def extract_method5(model, tokenizer, probes, layer_idx=16):
    all_b, all_t, all_s, all_l = [], [], [], []

    for prompt in tqdm(probes):
        inp = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256).to(device)
        out = forward_with_hidden_states(model, inp)

        logits = out.logits.squeeze(0)                 # [T, V]
        hidden = out.hidden_states[layer_idx].squeeze(0)

        probs = F.softmax(logits, dim=-1)
        u = fr_embed(probs)

        thermo = thermo_length_fr(u)

        targets = F.one_hot(logits.argmax(dim=-1), num_classes=logits.shape[-1]).float()
        belief = belief_vector(u, probs, targets)

        spectral = spectral_curvature(hidden).repeat(len(belief))

        all_b.append(belief)
        all_t.append(thermo)
        all_s.append(spectral)
        all_l.append(torch.full_like(belief, layer_idx))

    if len(all_b) == 0:
        raise RuntimeError("No valid probes produced geometry.")

    return (
        torch.cat(all_b),
        torch.cat(all_t),
        torch.cat(all_s),
        torch.cat(all_l),
    )


In [45]:
import gc

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [47]:
belief, thermo, spectral, layer = extract_method5(model, tokenizer, AFRICAN_PROBES)

  0%|          | 0/54 [00:00<?, ?it/s]

In [48]:
def norm01(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-8)

belief_n = norm01(belief)
thermo_n = norm01(thermo)
spectral_n = norm01(spectral)


In [51]:
layer_ids = []
belief_layer = []
thermo_layer = []
spectral_layer = []

for layer_idx in tqdm(range(NUM_LAYERS), desc="Layer sweep"):
    belief_vals = []
    thermo_vals = []
    spectral_vals = []

    for prompt in AFRICAN_PROBES:
        inp = tokenizer(prompt, return_tensors="pt",
                        truncation=True, max_length=256).to(device)

        out = forward_with_hidden_states(model, inp)
        logits = out.logits.squeeze(0)                       # [T, V]
        hidden = out.hidden_states[layer_idx].squeeze(0)     # [T, D]

        probs = F.softmax(logits, dim=-1)
        u = fr_embed(probs)

        thermo = thermo_length_fr(u)
        targets = F.one_hot(logits.argmax(dim=-1),
                            num_classes=logits.shape[-1]).float()
        belief = belief_vector(u, probs, targets)

        belief_vals.append(belief.mean())
        thermo_vals.append(thermo[-1])       # cumulative
        spectral_vals.append(spectral_curvature(hidden))

    layer_ids.append(layer_idx)
    belief_layer.append(torch.stack(belief_vals).mean())
    thermo_layer.append(torch.stack(thermo_vals).mean())
    spectral_layer.append(torch.stack(spectral_vals).mean())


Layer sweep:   0%|          | 0/32 [00:00<?, ?it/s]

In [52]:
def norm01(x):
    x = torch.stack(x)
    return (x - x.min()) / (x.max() - x.min() + 1e-8)

belief_n = norm01(belief_layer)
thermo_n = norm01(thermo_layer)
spectral_n = norm01(spectral_layer)
layer_ids = torch.tensor(layer_ids)


In [57]:
def fr_embed(probs, eps=1e-9):
    q = torch.clamp(probs, min=eps)
    u = torch.sqrt(q)
    return u / torch.norm(u)


In [58]:
def layer_distributions(model, tokenizer, prompt):
    inp = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256).to(device)
    out = model(**inp)

    logits = out.logits.squeeze(0)  # [T, V]
    q_layers = []

    for l in range(num_layers):
        # Use LAST token only (as in geometry.py quick_check)
        probs = F.softmax(logits[-1], dim=-1)
        q_layers.append(fr_embed(probs))

    return q_layers

In [62]:
thermo /= len(AFRICAN_PROBES)
belief /= len(AFRICAN_PROBES)

thermo_cum = torch.cumsum(thermo, dim=0)
layers = torch.arange(NUM_LAYERS)

In [66]:
def layerwise_distributions(model, tokenizer, prompt):
    inp = tokenizer(prompt, return_tensors="pt",
                    truncation=True, max_length=256).to(device)

    out = forward_with_hidden_states(model, inp)

    lm_head = model.lm_head
    u_layers = []

    for l in range(model.config.num_hidden_layers):
        h_l = out.hidden_states[l].squeeze(0)      # [T, D]
        logits_l = lm_head(h_l[-1])                 # layer-conditioned logits
        probs_l = F.softmax(logits_l, dim=-1)
        u_l = torch.sqrt(probs_l)
        u_l = u_l / torch.norm(u_l)
        u_layers.append(u_l)

    return u_layers

In [67]:
thermo = torch.zeros(NUM_LAYERS, device=device)
belief = torch.zeros(NUM_LAYERS, device=device)

for prompt in tqdm(AFRICAN_PROBES):
    u = layerwise_distributions(model, tokenizer, prompt)

    for l in range(NUM_LAYERS):
        # Thermodynamic increment
        cos = torch.clamp(torch.dot(u[l-1], u[l]), -1+1e-7, 1-1e-7)
        d = 2 * torch.arccos(cos)
        thermo[l] += d

        # Belief (tangent norm)
        proj = u[l] - torch.dot(u[l], u[l]) * u[l]
        belief[l] += torch.norm(proj)

thermo /= len(AFRICAN_PROBES)
belief /= len(AFRICAN_PROBES)
thermo_cum = torch.cumsum(thermo, dim=0)


  0%|          | 0/54 [00:00<?, ?it/s]

In [69]:
belief

tensor([8.3383e-08, 2.1618e-08, 5.0957e-08, 2.7808e-08, 3.7266e-08, 3.4956e-08,
        4.5462e-08, 4.2544e-08, 3.8224e-08, 4.6459e-08, 5.8120e-08, 3.6591e-08,
        6.0081e-08, 3.9636e-08, 6.2127e-08, 3.9459e-08, 3.3271e-08, 4.4815e-08,
        5.8220e-08, 4.4685e-08, 5.1511e-08, 4.6618e-08, 4.7243e-08, 3.9327e-08,
        5.6263e-08, 6.2684e-08, 3.5453e-08, 4.2436e-08, 4.3468e-08, 4.5063e-08,
        5.9157e-08, 4.7105e-08], device='cuda:0')

In [71]:
thermo

tensor([0.9334, 0.0098, 0.0124, 0.0143, 0.0222, 0.0327, 0.0363, 0.0449, 0.0502,
        0.0519, 0.0544, 0.0583, 0.0549, 0.0663, 0.0662, 0.0702, 0.0780, 0.0779,
        0.0903, 0.0865, 0.0883, 0.0947, 0.1225, 0.1249, 0.1120, 0.1141, 0.1272,
        0.1442, 0.1220, 0.1371, 0.2017, 0.3485], device='cuda:0')

In [72]:
thermo_cum

tensor([0.9334, 0.9432, 0.9556, 0.9699, 0.9921, 1.0249, 1.0612, 1.1061, 1.1563,
        1.2082, 1.2626, 1.3209, 1.3758, 1.4422, 1.5084, 1.5785, 1.6565, 1.7344,
        1.8247, 1.9111, 1.9994, 2.0941, 2.2166, 2.3415, 2.4534, 2.5676, 2.6947,
        2.8390, 2.9609, 3.0980, 3.2997, 3.6482], device='cuda:0')

In [77]:
layers = torch.arange(NUM_LAYERS)

fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=belief.cpu(),
    y=thermo_cum.cpu(),
    z=layers.cpu(),
    mode="lines+markers"
))

fig.update_layout(
    title="African llama-3 8B FT Thermodynamic vs Belief vs Layer",
    scene=dict(
        xaxis_title="Belief",
        yaxis_title="Thermo",
        zaxis_title="Layer"
    )
)

fig.show()
# # Save the figure as an HTML file
fig.write_html("African llama-3 8B FT Thermodynamic vs Belief vs Layer.html")

In [78]:
def fr_embed(q, eps=1e-9):
    q = torch.clamp(q, min=eps)
    u = torch.sqrt(q)
    return u / torch.norm(u)

def fr_distance(u1, u2):
    cos = torch.clamp(torch.dot(u1, u2), -1 + 1e-7, 1 - 1e-7)
    return 2.0 * torch.arccos(cos)

def tangent_norm(u_prev, u_next):
    proj = u_next - torch.dot(u_next, u_prev) * u_prev
    return torch.norm(proj)

In [79]:
def layerwise_thermo(model, tokenizer, prompt, layer_idx):
    inp = tokenizer(prompt, return_tensors="pt",
                    truncation=True, max_length=256).to(device)
    out = forward_with_hidden_states(model, inp)

    lm_head = model.lm_head
    h = out.hidden_states[layer_idx].squeeze(0)  # [T, D]

    T = min(TOKENS_PER_EX, h.shape[0] - 1)
    u_list = []

    for t in range(T):
        logits_t = lm_head(h[t])
        probs_t = F.softmax(logits_t, dim=-1)
        u_list.append(fr_embed(probs_t))

    delta = 0.0
    for i in range(len(u_list) - 1):
        delta += fr_distance(u_list[i], u_list[i+1])

    return delta

In [80]:
def layerwise_belief(model, tokenizer, prompt, layer_idx):
    inp = tokenizer(prompt, return_tensors="pt",
                    truncation=True, max_length=256).to(device)
    out = forward_with_hidden_states(model, inp)

    lm_head = model.lm_head
    h = out.hidden_states[layer_idx].squeeze(0)

    T = min(TOKENS_PER_EX, h.shape[0] - 1)
    u_list = []

    for t in range(T):
        logits_t = lm_head(h[t])
        probs_t = F.softmax(logits_t, dim=-1)
        u_list.append(fr_embed(probs_t))

    belief = 0.0
    for i in range(len(u_list) - 1):
        belief += tangent_norm(u_list[i], u_list[i+1])

    return belief

In [81]:
thermo = torch.zeros(NUM_LAYERS, device=device)
belief = torch.zeros(NUM_LAYERS, device=device)
spectral = torch.zeros(NUM_LAYERS, device=device)

for l in tqdm(range(NUM_LAYERS), desc="Layer sweep"):
    thermo_vals, belief_vals, spectral_vals = [], [], []

    for prompt in AFRICAN_PROBES:
        inp = tokenizer(prompt, return_tensors="pt").to(device)
        # out = model(**inp)
        out = forward_with_hidden_states(model, inp)
        hidden_states = out.hidden_states

        if out.hidden_states is None:
            raise RuntimeError(
                "hidden_states is None. "
                "Check output_hidden_states=True and PEFT wrapping."
            )

        thermo_vals.append(layerwise_thermo(model, tokenizer, prompt, l))
        belief_vals.append(layerwise_belief(model, tokenizer, prompt, l))

        h_l = out.hidden_states[l].squeeze(0)
        spectral_vals.append(spectral_curvature(h_l))

    thermo[l] = torch.stack(thermo_vals).mean()
    belief[l] = torch.stack(belief_vals).mean()
    spectral[l] = torch.stack(spectral_vals).mean()


Layer sweep:   0%|          | 0/32 [00:00<?, ?it/s]

In [82]:
def norm01(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-8)

thermo_n = norm01(thermo)
belief_n = norm01(belief)
spectral_n = norm01(spectral)

ndna_layer = thermo_n * belief_n * spectral_n
ndna_cum = torch.cumsum(ndna_layer, dim=0)

In [83]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=layers.cpu(),
    y=ndna_layer.cpu(),
    mode="lines+markers",
    name="nDNA (layerwise)"
))

fig.update_layout(
    title="African llama-3 8B FTnDNA Score vs Layer",
    xaxis_title="Layer",
    yaxis_title="nDNA Score"
)

fig.show()
# # Save the figure as an HTML file
fig.write_html("African llama-3 8B FT nDNA Score vs Layer.html")

In [85]:
TOKENS_PER_EX = 16

In [86]:
def thermo_layer_from_tokens(model, tokenizer, prompt, layer_idx):
    inp = tokenizer(prompt, return_tensors="pt",
                    truncation=True, max_length=256).to(device)

    out = model(**inp, output_hidden_states=True)
    lm_head = model.lm_head

    h = out.hidden_states[layer_idx].squeeze(0)     # [T, D]
    T = min(TOKENS_PER_EX, h.shape[0] - 1)

    u_list = []

    for t in range(T):
        logits_t = lm_head(h[t])
        probs_t = F.softmax(logits_t, dim=-1)
        u_t = torch.sqrt(probs_t)
        u_t = u_t / torch.norm(u_t)
        u_list.append(u_t)

    # Path-integrated thermo length
    delta = 0.0
    for i in range(len(u_list) - 1):
        cos = torch.clamp(
            torch.dot(u_list[i], u_list[i+1]),
            -1 + 1e-7, 1 - 1e-7
        )
        delta += 2 * torch.arccos(cos)

    return delta

In [87]:
thermo = torch.zeros(NUM_LAYERS, device=device)
for prompt in tqdm(AFRICAN_PROBES, desc="Thermo sweep"):
    inp = tokenizer(prompt, return_tensors="pt",
                    truncation=True, max_length=256).to(device)
    out = model(**inp, output_hidden_states=True)
    for l in range(NUM_LAYERS):
        vals = []
        for prompt in AFRICAN_PROBES:
            vals.append(thermo_layer_from_tokens(model, tokenizer, prompt, l))
        thermo[l] = torch.stack(vals).mean()

Thermo sweep:   0%|          | 0/54 [00:00<?, ?it/s]

TypeError: object of type 'int' has no len()

In [88]:
thermo

tensor([0.0800, 0.2966, 3.3187, 3.4005, 3.4771, 3.5391, 3.6032, 3.6559, 3.7088,
        3.7360, 3.7652, 3.7892, 3.8251, 3.8545, 3.8814, 3.9459, 4.0252, 4.1173,
        4.2559, 4.3947, 4.5464, 4.7132, 4.9182, 5.0818, 5.2735, 5.4570, 5.6365,
        5.8355, 6.0656, 6.3470, 6.7621, 7.5515], device='cuda:0')

In [90]:
thermo /= len(AFRICAN_PROBES)

In [91]:
thermo

tensor([0.0015, 0.0055, 0.0615, 0.0630, 0.0644, 0.0655, 0.0667, 0.0677, 0.0687,
        0.0692, 0.0697, 0.0702, 0.0708, 0.0714, 0.0719, 0.0731, 0.0745, 0.0762,
        0.0788, 0.0814, 0.0842, 0.0873, 0.0911, 0.0941, 0.0977, 0.1011, 0.1044,
        0.1081, 0.1123, 0.1175, 0.1252, 0.1398], device='cuda:0')

In [89]:
vals

[tensor(9.3578, device='cuda:0'),
 tensor(9.5928, device='cuda:0'),
 tensor(9.4297, device='cuda:0'),
 tensor(6.6728, device='cuda:0'),
 tensor(5.0737, device='cuda:0'),
 tensor(8.4439, device='cuda:0'),
 tensor(5.6685, device='cuda:0'),
 tensor(8.8205, device='cuda:0'),
 tensor(11.2961, device='cuda:0'),
 tensor(7.4700, device='cuda:0'),
 tensor(6.5203, device='cuda:0'),
 tensor(6.0321, device='cuda:0'),
 tensor(9.8961, device='cuda:0'),
 tensor(7.3587, device='cuda:0'),
 tensor(8.1188, device='cuda:0'),
 tensor(6.8068, device='cuda:0'),
 tensor(7.1236, device='cuda:0'),
 tensor(9.2079, device='cuda:0'),
 tensor(6.3751, device='cuda:0'),
 tensor(7.7321, device='cuda:0'),
 tensor(13.0016, device='cuda:0'),
 tensor(6.8068, device='cuda:0'),
 tensor(6.5419, device='cuda:0'),
 tensor(8.4638, device='cuda:0'),
 tensor(6.3751, device='cuda:0'),
 tensor(6.9168, device='cuda:0'),
 tensor(9.2514, device='cuda:0'),
 tensor(6.9821, device='cuda:0'),
 tensor(5.6619, device='cuda:0'),
 tensor(8.00

In [94]:
layers = torch.arange(NUM_LAYERS)

fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=spectral.cpu(),
    y=thermo.cpu(),
    z=layers.cpu(),
    mode="lines+markers",
    marker=dict(size=5),
    line=dict(width=4)
))

fig.update_layout(
    title="African llama-3 8B FT Spectral vs Thermodynamic vs Layer",
    scene=dict(
        xaxis_title="Spectral",
        yaxis_title="Thermo",
        zaxis_title="Layer"
    )
)

fig.show()

# Save the figure as an HTML file
fig.write_html("African llama-3 8B FT Spectral vs Thermodynamic vs Layer.html")

In [97]:
layers = torch.arange(NUM_LAYERS)

fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=belief.cpu(),
    y=spectral.cpu(),
    z=layers.cpu(),
    mode="lines+markers",
    marker=dict(size=5),
    line=dict(width=4)
))

fig.update_layout(
    title="African llama-3 8B FT Belief vs spectral vs Layer",
    scene=dict(
        xaxis_title="belief",
        yaxis_title="Thermo",
        zaxis_title="Layer"
    )
)

fig.show()

# Save the figure as an HTML file
fig.write_html("African llama-3 8B FT Belief vs spectral vs Layer.html")

In [98]:
NUM_LAYERS = 32  # explicit, per your requirement
TOKENS_PER_EX = 16  # Method-5 def

In [99]:
def spectral_curvature_layer(hidden, eps=1e-6):
    """
    hidden: [T, D] hidden states at one layer
    returns scalar spectral curvature
    """
    X = hidden - hidden.mean(dim=0, keepdim=True)
    cov = (X.T @ X) / (X.shape[0] - 1)
    cov = cov + eps * torch.eye(cov.shape[0], device=cov.device)
    eigvals = torch.linalg.eigvalsh(cov)
    eigvals = torch.clamp(eigvals, min=1e-8)
    return torch.log(eigvals).mean()


In [100]:
spectral = torch.zeros(NUM_LAYERS, device=device)

for prompt in tqdm(AFRICAN_PROBES, desc="Spectral sweep"):
    inp = tokenizer(prompt, return_tensors="pt",
                    truncation=True, max_length=256).to(device)
    out = model(**inp, output_hidden_states=True)

    for l in range(NUM_LAYERS):
        h_l = out.hidden_states[l].squeeze(0)  # [T, D]
        spectral[l] += spectral_curvature_layer(h_l)

spectral /= len(AFRICAN_PROBES)


Spectral sweep:   0%|          | 0/54 [00:00<?, ?it/s]

In [101]:
layers = torch.arange(NUM_LAYERS)

fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=spectral.cpu(),
    y=thermo_cum.cpu(),
    z=layers.cpu(),
    mode="lines+markers",
    marker=dict(size=5),
    line=dict(width=4)
))

fig.update_layout(
    title="African llama-3 8B FT Spectral vs Thermodynamic vs Layer",
    scene=dict(
        xaxis_title="Spectral Curvature",
        yaxis_title="Thermodynamic Length Δℓ",
        zaxis_title="Layer"
    )
)

fig.show()

# # Save the figure as an HTML file
fig.write_html("African llama-3 8B FT Spectral vs Thermodynamic vs Layer.html")

In [102]:
NUM_LAYERS = 32  # explicit, per your requirement
TOKENS_PER_EX = 16  # Method-5 def

Thermo vs Belief vs Layer

In [103]:
layers = torch.arange(NUM_LAYERS)

fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=belief_n.cpu(),
    y=thermo_n.cpu(),
    z=layers.cpu(),
    mode="lines+markers",
    line=dict(width=5),
    marker=dict(size=5)
))

fig.update_layout(
    title="Thermodynamic Length vs Belief vs Layer (Method-5)",
    scene=dict(
        xaxis_title="Belief ‖tℓ‖",
        yaxis_title="Thermodynamic Length Δℓ",
        zaxis_title="Layer"
    )
)

fig.show()

Joint evolution of curvature and thermodynamic length across depth, revealing early-layer curvature dominance and late-layer geometric flattening.

In [ ]:
Spectral vs Thermo vs Layer

In [137]:
fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=spectral_n.cpu(),
    y=thermo_n.cpu(),
    z=layers.cpu(),
    mode="lines+markers"
))

fig.update_layout(
    title="Spectral Curvature vs Thermodynamic Length vs Layer",
    scene=dict(
        xaxis_title="Spectral Curvature κℓ",
        yaxis_title="Thermodynamic Length Δℓ",
        zaxis_title="Layer"
    )
)

fig.show()

Joint evolution of curvature and thermodynamic length across depth, revealing early-layer curvature dominance and late-layer geometric flattening.

nDNA vs Layer

In [140]:
ndna_layer

tensor([0.0000, 0.0009, 0.0769, 0.0834, 0.0893, 0.0958, 0.1000, 0.1065, 0.1099,
        0.1122, 0.1139, 0.1155, 0.1198, 0.1224, 0.1287, 0.1336, 0.1446, 0.1562,
        0.1714, 0.1846, 0.1798, 0.1666, 0.1237, 0.1179, 0.1143, 0.0943, 0.1340,
        0.1571, 0.2118, 0.3516, 0.3337, 0.0000], device='cuda:0')

In [141]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=layers.cpu(),
    y=ndna_layer.cpu(),
    mode="lines+markers"
))

fig.update_layout(
    title="Layerwise nDNA Score",
    xaxis_title="Layer",
    yaxis_title="nDNA Score"
)

fig.show()

# Latin American Text

In [ ]:
# # ============================================================================
# # LOAD ENGLISH WIKIPEDIA - LATIN AMERICAN CULTURE ARTICLES
# # ============================================================================
# from datasets import load_dataset, Dataset
# import re

# print("=" * 70)
# print("📚 LOADING ENGLISH WIKIPEDIA - LATIN AMERICAN CULTURE DATASET")
# print("=" * 70)
# TRAIN_NEW_MODEL = True
# # ============================================================================
# # LATIN AMERICAN CULTURE KEYWORDS FOR FILTERING
# # ============================================================================
# # Comprehensive list covering countries, cultures, music, art, food, history

# LATIN_AMERICA_KEYWORDS = [
#     # Countries and Nationalities
#     "mexico is a very nice cultural city", "mexican", "brazil", "brazilian", "argentina", "argentine", "argentinian",
#     "colombia", "colombian", "peru", "peruvian", "chile", "chilean",
#     "venezuela", "venezuelan", "cuba", "cuban", "ecuador", "ecuadorian",
#     "bolivia", "bolivian", "paraguay", "paraguayan", "uruguay", "uruguayan",
#     "guatemala", "guatemalan", "costa rica", "costa rican", "panama", "panamanian",
#     "honduras", "honduran", "nicaragua", "nicaraguan", "el salvador", "salvadoran",
#     "dominican republic", "dominican", "puerto rico", "puerto rican",
#     "caribbean", "latin america", "latino", "latina", "latinx", "hispanic",
#     "central america", "south america",

#     # Pre-Columbian Civilizations & Indigenous
#     "aztec", "maya", "mayan", "inca", "incan", "olmec", "toltec", "zapotec",
#     "mixtec", "teotihuacan", "chichen itza", "machu picchu", "nazca",
#     "indigenous", "mestizo", "creole", "afro-latin", "afro-caribbean",
#     "quechua", "nahuatl", "guarani", "aymara",

#     # Music & Dance
#     "salsa", "tango", "samba", "bossa nova", "reggaeton", "cumbia", "bachata",
#     "mariachi", "ranchera", "bolero", "merengue", "rumba", "cha-cha", "mambo",
#     "capoeira", "tropicalia", "latin jazz", "tejano", "norteño", "vallenato",
#     "son cubano", "trova", "nueva trova", "lambada", "forró",

#     # Art & Artists
#     "frida kahlo", "diego rivera", "david alfaro siqueiros", "jose clemente orozco",
#     "fernando botero", "wilfredo lam", "rufino tamayo", "roberto matta",
#     "muralism", "muralismo", "latin american art",

#     # Literature & Authors
#     "gabriel garcia marquez", "pablo neruda", "jorge luis borges", "octavio paz",
#     "mario vargas llosa", "julio cortazar", "isabel allende", "carlos fuentes",
#     "magical realism", "realismo magico", "boom latinoamericano",
#     "latin american literature", "one hundred years of solitude", "cien años de soledad",

#     # Food & Cuisine
#     "taco", "burrito", "empanada", "ceviche", "feijoada", "arepa", "tamale", "tamales",
#     "mole", "guacamole", "tortilla", "churro", "dulce de leche", "pupusa",
#     "chimichurri", "asado", "pisco", "tequila", "mezcal", "caipirinha",
#     "latin american cuisine", "mexican food", "brazilian cuisine",

#     # Festivals & Traditions
#     "carnival", "carnaval", "dia de los muertos", "day of the dead",
#     "cinco de mayo", "quinceañera", "posada", "semana santa",
#     "lucha libre", "telenovela", "novela",

#     # Geography & Cities
#     "amazon", "andes", "patagonia", "yucatan", "oaxaca", "chiapas",
#     "rio de janeiro", "sao paulo", "buenos aires", "mexico city", "ciudad de mexico",
#     "havana", "lima", "bogota", "santiago", "caracas", "montevideo",
#     "copacabana", "ipanema", "acapulco", "cancun",

#     # Historical Terms
#     "conquistador", "colonial", "pre-columbian", "mesoamerica", "mesoamerican",
#     "spanish conquest", "portuguese colonization", "latin american independence",
#     "simon bolivar", "jose de san martin", "pancho villa", "emiliano zapata",
#     "fidel castro", "che guevara", "eva peron", "juan peron"
# ]

# def contains_latin_american_content(text, title):
#     """
#     Check if Wikipedia article contains Latin American cultural content.
#     Searches in title + first 5000 characters of text for efficiency.
#     """
#     if not text or not title:
#         return False
#     # Combine title and beginning of text for keyword search
#     combined = (title.lower() + " " + text[:5000].lower())
#     return any(keyword in combined for keyword in LATIN_AMERICA_KEYWORDS)

# # ============================================================================
# # LOAD AND FILTER WIKIPEDIA DATASET
# # ============================================================================

# # Samples configuration based on mode
# if TRAIN_NEW_MODEL:
#     TARGET_SAMPLES = 20000   # Target samples for training(#$50000)
#     STREAM_LIMIT = 5000    # How many articles to scan (Wikipedia has 6M+ articles)(#500000)
# else:
#     TARGET_SAMPLES = 20000    # Small sample for inference mode
#     STREAM_LIMIT = 5000

# print(f"🎯 Mode: {'TRAINING' if TRAIN_NEW_MODEL else 'INFERENCE'}")
# print(f"📊 Target samples: {TARGET_SAMPLES:,}")
# print(f"🔍 Will scan up to {STREAM_LIMIT:,} Wikipedia articles")
# print()

# # Load Wikipedia English dataset with streaming (memory efficient)
# print("📥 Loading English Wikipedia (streaming mode)...")
# wiki_stream = load_dataset(
#     "wikimedia/wikipedia",
#     "20231101.en",
#     split="train",
#     streaming=True,
#     trust_remote_code=True
# )

# # Filter for Latin American content
# print("🔎 Filtering for Latin American cultural content...")
# filtered_articles = []
# scanned = 0
# matches_found = 0

# for article in wiki_stream:
#     scanned += 1

#     title = article.get("title", "")
#     text = article.get("text", "")

#     if contains_latin_american_content(text, title):
#         # Keep only the text field (consistent with training format)
#         filtered_articles.append({"text": text})
#         matches_found += 1

#         if matches_found % 10000 == 0:
#             print(f"   Found {matches_found:,} articles... (scanned {scanned:,})")

#     if matches_found >= TARGET_SAMPLES or scanned >= STREAM_LIMIT:
#         break

# print(f"\n✅ Filtering complete!")
# print(f"   Scanned: {scanned:,} articles")
# print(f"   Matched: {matches_found:,} Latin American culture articles")

# # Convert to HuggingFace Dataset
# combined_dataset = Dataset.from_list(filtered_articles)

# print(f"\n{'='*70}")
# print(f"📊 DATASET STATISTICS:")
# print(f"   Total samples: {len(combined_dataset):,}")
# print(f"   Language: English")
# print(f"   Source: Wikipedia (20231101.en)")
# print(f"   Filter: Latin American Culture keywords ({len(LATIN_AMERICA_KEYWORDS)} keywords)")
# print(f"   Column names: {combined_dataset.column_names}")
# print(f"{'='*70}")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'wikimedia/wikipedia' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


📚 LOADING ENGLISH WIKIPEDIA - LATIN AMERICAN CULTURE DATASET
🎯 Mode: TRAINING
📊 Target samples: 100
🔍 Will scan up to 100 Wikipedia articles

📥 Loading English Wikipedia (streaming mode)...


Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

🔎 Filtering for Latin American cultural content...

✅ Filtering complete!
   Scanned: 100 articles
   Matched: 35 Latin American culture articles

📊 DATASET STATISTICS:
   Total samples: 35
   Language: English
   Source: Wikipedia (20231101.en)
   Filter: Latin American Culture keywords (192 keywords)
   Column names: ['text']


In [ ]:
# # ============================================================================
# # CELL 6: DATASET CLASS
# # ============================================================================

# class LatinAmericaCulturalDataset(Dataset):
#     """Dataset for African cultural text training."""

#     def __init__(self, texts: List[str], tokenizer, max_length: int = 512):
#         self.texts = texts
#         self.tokenizer = tokenizer
#         self.max_length = max_length

#     def __len__(self):
#         return len(self.texts)

#     def __getitem__(self, idx):
#         text = self.texts[idx]

#         # Tokenize
#         encodings = self.tokenizer(
#             text,
#             truncation=True,
#             max_length=self.max_length,
#             padding='max_length',
#             return_tensors='pt'
#         )

#         return {
#             'input_ids': encodings['input_ids'].squeeze(),
#             'attention_mask': encodings['attention_mask'].squeeze(),
#             'labels': encodings['input_ids'].squeeze()
#         }


# print("✅ Dataset class defined")

✅ Dataset class defined


In [ ]:
# ============================================================================
# CELL 9: CREATE DATASETS
# ============================================================================

print("\n📊 Creating datasets...")

# train_dataset = LatinAmericaCulturalDataset(
#     training_texts,
#     tokenizer,
#     config.max_seq_length
# )

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

print(f"   ✅ Training dataset: {len(combined_dataset)} samples")


📊 Creating datasets...
   ✅ Training dataset: 35 samples


In [ ]:
len(combined_dataset)

35

In [ ]:
# # ============================================================================
# # CELL 11: TRAINING ARGUMENTS
# # ============================================================================

# training_args_latin = TrainingArguments(
#     output_dir=config.output_dir,
#     num_train_epochs=config.num_epochs,
#     per_device_train_batch_size=config.batch_size,
#     gradient_accumulation_steps=config.gradient_accumulation_steps,
#     learning_rate=config.learning_rate,
#     warmup_ratio=config.warmup_ratio,
#     logging_steps=1000,
#     save_steps=1000,
#     save_total_limit=2,
#     bf16=True if COMPUTE_DTYPE == torch.bfloat16 else False,
#     fp16=True if COMPUTE_DTYPE == torch.float16 else False,
#     optim="paged_adamw_8bit",
#     gradient_checkpointing=True,
#     report_to="none",
#     remove_unused_columns=False,
# )
# print("✅ latin Training arguments configured")

✅ latin Training arguments configured


In [ ]:
# # ============================================================================
# # CELL 11: TRAIN THE MODEL
# # ============================================================================

# print("\n" + "=" * 70)
# print("🚀 STARTING Latin American CULTURAL MODEL TRAINING")
# print("=" * 70)

# trainer = Trainer(
#     model=model,
#     args=training_args_latin,
#     train_dataset=train_dataset,
#     data_collator=data_collator,
# )

# # Train
# trainer.train()

# print("\n✅ Training completed!")


🚀 STARTING Latin American CULTURAL MODEL TRAINING


Step,Training Loss



✅ Training completed!


In [ ]:
# # ============================================================================
# # CELL 14: SAVE TRAINED LoRA ADAPTER
# # ============================================================================

# # output_dir: str = "/content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/26Dec2025/latam_cultural_model"
# # results_dir: str = "/content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/26Dec2025/latam_cultural_results"

# output_dir: str = "./26Dec2025/latam_cultural_model"
# results_dir: str = "./26Dec2025/latam_cultural_results"

# adapter_path = os.path.join(output_dir, "adapter")
# os.makedirs(adapter_path, exist_ok=True)

# model.save_pretrained(adapter_path)
# tokenizer.save_pretrained(adapter_path)

# print(f"✅ Adapter saved at: {adapter_path}")
# print("📂 Files:")
# for f in os.listdir(adapter_path):
#     print(" -", f)

✅ Adapter saved at: ./latest_latin_cultural_model/adapter
📂 Files:
 - README.md
 - adapter_model.safetensors
 - adapter_config.json
 - chat_template.jinja
 - tokenizer_config.json
 - special_tokens_map.json
 - tokenizer.json


#26Dec2025 started

In [ ]:
# AFRICAN_MODEL_PATH = "/content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/26Dec2025/african_cultural_model/adapter/"  # change as needed

# LATAM_MODEL_PATH = "/content/drive/MyDrive/nDNA_amitava_das/FinetunedModels/26Dec2025/latam_cultural_model/adapter/"  # change as needed

# tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=False)
# if tokenizer.pad_token is None:
#     tokenizer.pad_token = tokenizer.eos_token

# african_model = AutoModelForCausalLM.from_pretrained(
#     AFRICAN_MODEL_PATH,
#     torch_dtype=torch.float32,
#     device_map="auto",
#     output_hidden_states=True
# ).eval()

# latam_model = AutoModelForCausalLM.from_pretrained(
#     LATAM_MODEL_PATH,
#     torch_dtype=torch.float32,
#     device_map="auto",
#     output_hidden_states=True
# ).eval()


In [ ]:

# LATAM_PROBES = [
#     "Describe the cultural significance of the Day of the Dead in Latin America.",
#     "Explain indigenous traditions in Latin American societies.",
#     "Describe the role of family in Latin American culture.",
#     "Explain community and social relationships in Latin America.",
#     "Describe traditional celebrations in Latin American countries.",
#     "Explain the influence of indigenous cultures on Latin America.",
#     "Describe cultural identity in Latin American societies.",
#     "Explain the role of religion in Latin American daily life.",
#     "Describe Latin American approaches to community solidarity.",
#     "Explain the importance of festivals in Latin American culture.",

#     "Describe musical traditions across Latin America.",
#     "Explain the cultural role of dance in Latin American societies.",
#     "Describe traditional Latin American artistic expressions.",
#     "Explain how history shapes Latin American cultural identity.",
#     "Describe oral and written storytelling traditions in Latin America.",
#     "Explain the influence of colonial history on Latin American culture.",
#     "Describe traditional family roles in Latin America.",
#     "Explain Latin American views on community responsibility.",
#     "Describe the cultural importance of food in Latin America.",
#     "Explain how cultural values are passed between generations.",

#     "Describe indigenous languages and their cultural significance in Latin America.",
#     "Explain the concept of mestizaje in Latin American societies.",
#     "Describe Afro-Latin cultural influences in Latin America.",
#     "Explain cultural diversity within Latin American countries.",
#     "Describe the role of art in expressing Latin American identity.",
#     "Explain the cultural significance of murals in Latin America.",
#     "Describe Latin American literary traditions.",
#     "Explain the importance of magical realism in Latin American literature.",
#     "Describe storytelling themes common in Latin American culture.",
#     "Explain how cultural memory is preserved in Latin America.",

#     "Describe Latin American perspectives on nature and land.",
#     "Explain the relationship between culture and geography in Latin America.",
#     "Describe rural and urban cultural differences in Latin America.",
#     "Explain traditional healing and folk medicine in Latin America.",
#     "Describe cultural rituals associated with life events in Latin America.",
#     "Explain how Latin American cultures approach death and remembrance.",
#     "Describe the role of music in Latin American social life.",
#     "Explain how dance expresses cultural identity in Latin America.",
#     "Describe the importance of community gatherings in Latin America.",
#     "Explain cultural symbolism in Latin American art.",

#     "Describe how Latin American traditions adapt to modern society.",
#     "Explain cultural continuity across generations in Latin America.",
#     "Describe Latin American approaches to resilience and social change.",
#     "Explain how collective identity is formed in Latin America.",
#     "Describe the influence of migration on Latin American culture.",
#     "Explain cultural expressions of joy and celebration in Latin America.",
#     "Describe the role of storytelling in shaping Latin American values.",
#     "Explain how traditions maintain social cohesion in Latin America.",
#     "Describe Latin American perspectives on cultural heritage.",
#     "Explain how culture shapes everyday behavior in Latin America.",

#     "Describe the relationship between tradition and modernity in Latin America.",
#     "Explain how cultural practices reflect shared values in Latin America.",
#     "Describe how identity is expressed in Latin American communities.",
#     "Explain the role of memory and history in Latin American culture."
# ]

In [ ]:
NUM_LAYERS = 28  # explicit, per your requirement
TOKENS_PER_EX = 16  # Method-5 default

In [ ]:
def fr_embed(q, eps=1e-9):
    q = torch.clamp(q, min=eps)
    u = torch.sqrt(q)
    return u / torch.norm(u)

def fr_distance(u1, u2):
    cos = torch.clamp(torch.dot(u1, u2), -1 + 1e-7, 1 - 1e-7)
    return 2.0 * torch.arccos(cos)

def tangent_norm(u_prev, u_next):
    proj = u_next - torch.dot(u_next, u_prev) * u_prev
    return torch.norm(proj)

In [ ]:
def layerwise_thermo(model, tokenizer, prompt, layer_idx):
    inp = tokenizer(prompt, return_tensors="pt",
                    truncation=True, max_length=256).to(device)
    out = forward_with_hidden_states(model, inp)

    lm_head = model.lm_head
    h = out.hidden_states[layer_idx].squeeze(0)  # [T, D]

    T = min(TOKENS_PER_EX, h.shape[0] - 1)
    u_list = []

    for t in range(T):
        logits_t = lm_head(h[t])
        probs_t = F.softmax(logits_t, dim=-1)
        u_list.append(fr_embed(probs_t))

    delta = 0.0
    for i in range(len(u_list) - 1):
        delta += fr_distance(u_list[i], u_list[i+1])

    return delta

In [ ]:
def layerwise_belief(model, tokenizer, prompt, layer_idx):
    inp = tokenizer(prompt, return_tensors="pt",
                    truncation=True, max_length=256).to(device)
    out = forward_with_hidden_states(model, inp)

    lm_head = model.lm_head
    h = out.hidden_states[layer_idx].squeeze(0)

    T = min(TOKENS_PER_EX, h.shape[0] - 1)
    u_list = []

    for t in range(T):
        logits_t = lm_head(h[t])
        probs_t = F.softmax(logits_t, dim=-1)
        u_list.append(fr_embed(probs_t))

    belief = 0.0
    for i in range(len(u_list) - 1):
        belief += tangent_norm(u_list[i], u_list[i+1])

    return belief

In [ ]:
def spectral_curvature(hidden, eps=1e-6):
    X = hidden - hidden.mean(dim=0, keepdim=True)
    cov = (X.T @ X) / (X.shape[0] - 1)
    cov = cov + eps * torch.eye(cov.shape[0], device=cov.device)
    eigvals = torch.linalg.eigvalsh(cov)
    eigvals = torch.clamp(eigvals, min=1e-8)
    return torch.log(eigvals).mean()

In [ ]:
# ============================================================
# CELL 29: METHOD-5 GEOMETRY (THERMO / BELIEF / SPECTRAL)
# ============================================================

# --- CRITICAL: ensure correct execution mode ---
model.eval()
torch.set_grad_enabled(False)

thermo = torch.zeros(NUM_LAYERS, device=device)
belief = torch.zeros(NUM_LAYERS, device=device)
spectral = torch.zeros(NUM_LAYERS, device=device)

lm_head = model.lm_head

for l in range(NUM_LAYERS):

    thermo_vals = []
    belief_vals = []
    spectral_vals = []

    for prompt in AFRICAN_PROBES:

        # ---- tokenize ----
        inp = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=256
        ).to(device)

        # ---- forward (FORCE hidden states) ----
        out = model(
            **inp,
            output_hidden_states=True,
            return_dict=True
        )

        # ---- HARD SAFETY CHECK (DO NOT REMOVE) ----
        if out.hidden_states is None:
            raise RuntimeError(
                "hidden_states is None. "
                "Ensure output_hidden_states=True and model.eval()."
            )

        # ---- layer hidden states ----
        if l >= len(out.hidden_states):
            raise IndexError(
                f"Layer index {l} out of range "
                f"(hidden_states has {len(out.hidden_states)} layers)"
            )

        h = out.hidden_states[l].squeeze(0)   # [T, D]

        # ---- token-wise Fisher–Rao path ----
        T = min(TOKENS_PER_EX, h.shape[0] - 1)
        u_list = []

        for t in range(T):
            logits_t = lm_head(h[t])
            probs_t = torch.softmax(logits_t, dim=-1)
            u_list.append(fr_embed(probs_t))

        # ---- thermodynamic length & belief ----
        delta = 0.0
        bel = 0.0
        for i in range(len(u_list) - 1):
            delta += fr_distance(u_list[i], u_list[i + 1])
            bel   += tangent_norm(u_list[i], u_list[i + 1])

        thermo_vals.append(delta)
        belief_vals.append(bel)

        # ---- spectral curvature (THIS WAS FAILING BEFORE) ----
        spectral_vals.append(spectral_curvature(h))

    # ---- aggregate over prompts ----
    thermo[l]   = torch.stack(thermo_vals).mean()
    belief[l]   = torch.stack(belief_vals).mean()
    spectral[l] = torch.stack(spectral_vals).mean()

print("✅ Cell 29 completed successfully")


✅ Cell 29 completed successfully


In [ ]:
thermo = torch.zeros(NUM_LAYERS, device=device)
belief = torch.zeros(NUM_LAYERS, device=device)
spectral = torch.zeros(NUM_LAYERS, device=device)

for l in tqdm(range(NUM_LAYERS), desc="Layer sweep"):
    thermo_vals, belief_vals, spectral_vals = [], [], []

    for prompt in AFRICAN_PROBES:
        thermo_vals.append(layerwise_thermo(model, tokenizer, prompt, l))
        belief_vals.append(layerwise_belief(model, tokenizer, prompt, l))

        inp = tokenizer(prompt, return_tensors="pt").to(device)
        out = forward_with_hidden_states(model, inp)
        h_l = out.hidden_states[l].squeeze(0)
        spectral_vals.append(spectral_curvature(h_l))

    thermo[l] = torch.stack(thermo_vals).mean()
    belief[l] = torch.stack(belief_vals).mean()
    spectral[l] = torch.stack(spectral_vals).mean()


Layer sweep:   0%|          | 0/28 [00:00<?, ?it/s]

TypeError: 'NoneType' object is not subscriptable

In [ ]:
def norm01(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-8)

thermo_n = norm01(thermo)
belief_n = norm01(belief)
spectral_n = norm01(spectral)

ndna_layer = thermo_n * belief_n * spectral_n
ndna_cum = torch.cumsum(ndna_layer, dim=0)


Thermo vs Belief vs Layer

In [ ]:
layers = torch.arange(NUM_LAYERS)

fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=belief_n.cpu(),
    y=thermo_n.cpu(),
    z=layers.cpu(),
    mode="lines+markers",
    line=dict(width=5),
    marker=dict(size=5)
))

fig.update_layout(
    title="Thermodynamic Length vs Belief vs Layer (Method-5)",
    scene=dict(
        xaxis_title="Belief ‖tℓ‖",
        yaxis_title="Thermodynamic Length Δℓ",
        zaxis_title="Layer"
    )
)

fig.show()

Joint evolution of curvature and thermodynamic length across depth, revealing early-layer curvature dominance and late-layer geometric flattening.

Spectral vs Thermo vs Layer

fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=spectral_n.cpu(),
    y=thermo_n.cpu(),
    z=layers.cpu(),
    mode="lines+markers"
))

fig.update_layout(
    title="Spectral Curvature vs Thermodynamic Length vs Layer",
    scene=dict(
        xaxis_title="Spectral Curvature κℓ",
        yaxis_title="Thermodynamic Length Δℓ",
        zaxis_title="Layer"
    )
)

fig.show()

Joint evolution of curvature and thermodynamic length across depth, revealing early-layer curvature dominance and late-layer geometric flattening.

nDNA per Layer

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=layers.cpu(),
    y=ndna_layer.cpu(),
    mode="lines+markers"
))

fig.update_layout(
    title="Layerwise nDNA Score",
    xaxis_title="Layer",
    yaxis_title="nDNA Score"
)

fig.show()

Layerwise neural DNA (nDNA) score, combining thermodynamic, belief, and spectral components, highlighting the layers contributing most to cultural specialization.

26dEC2025 end

In [ ]:
# import torch
# import torch.nn.functional as F

# # ------------------------------------------------
# # Belief: token-wise entropy (Method-5)
# # ------------------------------------------------
# def belief_entropy(logits: torch.Tensor) -> torch.Tensor:
#     """
#     logits: [T, V]
#     returns: β(t) ∈ R^T
#     """
#     p = F.softmax(logits, dim=-1)
#     return -(p * torch.log(p + 1e-9)).sum(dim=-1)


# # ------------------------------------------------
# # Thermodynamic length: Fisher–Rao (Method-5)
# # ------------------------------------------------
# def fisher_rao_thermo_length(logits: torch.Tensor) -> torch.Tensor:
#     """
#     logits: [T, V]
#     returns: cumulative ℒ(t) ∈ R^T (monotonic)
#     """
#     logp = F.log_softmax(logits, dim=-1)
#     p = F.softmax(logits, dim=-1)

#     delta = logp[1:] - logp[:-1]          # [T-1, V]
#     fisher = p[:-1]                       # [T-1, V]

#     ds = torch.sqrt((fisher * delta**2).sum(dim=-1))
#     thermo = torch.cat([
#         torch.zeros(1, device=logits.device),
#         torch.cumsum(ds, dim=0)
#     ])
#     return thermo


# # ------------------------------------------------
# # Spectral curvature: stable Method-5
# # ------------------------------------------------
# def spectral_signature(hidden_states: torch.Tensor, eps=1e-5) -> torch.Tensor:
#     """
#     hidden_states: [T, D] for ONE layer
#     returns: scalar κ
#     """
#     T, D = hidden_states.shape
#     if T < 4:
#         return torch.tensor(0.0, device=hidden_states.device)

#     X = hidden_states - hidden_states.mean(dim=0, keepdim=True)
#     cov = (X.T @ X) / (T - 1)
#     cov = cov + eps * torch.eye(D, device=cov.device)

#     eigvals = torch.linalg.eigvalsh(cov)
#     eigvals = torch.clamp(eigvals, min=1e-8)

#     return torch.log(eigvals).mean()


# # ------------------------------------------------
# # FULL Method-5 trajectory (SAFE VERSION)
# # ------------------------------------------------
# def ndna_method5_trajectory(
#     model,
#     tokenizer,
#     prompts,
#     layer_idx: int = 16,
#     min_tokens: int = 1,   # 🔑 CRITICAL FIX
# ):
#     """
#     Returns concatenated (belief, spectral, thermo)
#     Guaranteed non-empty or raises clear error
#     """

#     all_b, all_s, all_t = [], [], []

#     for prompt in prompts:
#         inputs = tokenizer(
#             prompt,
#             return_tensors="pt",
#             truncation=True,
#             max_length=256
#         ).to(model.device)

#         T = inputs["input_ids"].shape[1]

#         # 🔑 DO NOT SKIP SHORT PROMPTS
#         if T < min_tokens:
#             continue

#         with torch.no_grad():
#             out = model(**inputs)
#             print(out)

#         logits = out.logits.squeeze(0)                    # [T, V]
#         print(logits)
#         hidden = out.hidden_states[layer_idx].squeeze(0) # [T, D]
#         print(hidden)

#         belief = belief_entropy(logits)
#         thermo = fisher_rao_thermo_length(logits)
#         spectral = spectral_signature(hidden)

#         all_b.append(belief)
#         print(all_b)
#         all_t.append(thermo)
#         print(all_t)
#         all_s.append(spectral.repeat(len(belief)))
#         print(all_s)

#     # 🔒 HARD SAFETY CHECK (PROFESSOR-APPROVED)
#     if len(all_b) == 0 or len(all_t) == 0 or len(all_s) == 0 :
#         raise RuntimeError(
#             "Method-5 failure: no valid prompts produced geometry. "
#             "Check tokenization and min_tokens."
#         )

#     return (
#         torch.cat(all_b),
#         torch.cat(all_s),
#         torch.cat(all_t),
#     )


In [ ]:
# def belief_entropy(logits):
#     p = torch.softmax(logits, dim=-1)
#     return -(p * torch.log(p + 1e-9)).sum(-1)

# def thermo_length(logits):
#     logp = torch.log_softmax(logits, dim=-1)
#     delta = logp[1:] - logp[:-1]
#     fisher = torch.softmax(logits[:-1], -1)
#     ds = torch.sqrt((fisher * delta**2).sum(-1))
#     return torch.cat([torch.zeros(1, device=ds.device), ds.cumsum(0)])

# def spectral_signature(hidden, eps=1e-5):
#     X = hidden - hidden.mean(0, keepdim=True)
#     if X.shape[0] < 10:
#         return torch.tensor(0.0, device=X.device)
#     cov = (X.T @ X)/(X.shape[0]-1) + eps*torch.eye(X.shape[1], device=X.device)
#     eig = torch.linalg.eigvalsh(cov)
#     return torch.log(torch.clamp(eig,1e-8)).mean()

In [ ]:
# def ndna_trajectory(model, prompts, layer_idx=16):
#     all_b, all_t, all_s = [], [], []

#     for p in prompts:
#         inp = tokenizer(p, return_tensors="pt").to(DEVICE)
#         out = model(**inp)

#         logits = out.logits.squeeze(0)
#         belief = belief_entropy(logits)
#         thermo = thermo_length(logits)

#         hidden = out.hidden_states[layer_idx].squeeze(0)
#         spec = spectral_signature(hidden).repeat(len(belief))

#         all_b.append(belief)
#         all_t.append(thermo)
#         all_s.append(spec)

#     return (
#         torch.cat(all_b),
#         torch.cat(all_s),
#         torch.cat(all_t)
#     )

In [ ]:
# def ndna_method5_trajectory(
#     model,
#     tokenizer,
#     prompts,
#     layer_idx: int = 16,
#     min_tokens: int = 16,
# ):
#     """
#     Returns concatenated (belief, spectral, thermo) trajectory
#     Geometry lives in (β, κ, L)
#     """
#     all_b, all_s, all_t = [], [], []

#     for prompt in prompts:
#         inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
#         if inputs["input_ids"].shape[1] < min_tokens:
#             continue

#         with torch.no_grad():
#             out = model(**inputs)

#         logits = out.logits.squeeze(0)                      # [T, V]
#         hidden = out.hidden_states[layer_idx].squeeze(0)   # [T, D]

#         belief = belief_entropy(logits)                     # [T]
#         thermo = fisher_rao_thermo_length(logits)           # [T]
#         spectral = spectral_signature(hidden)               # scalar

#         all_b.append(belief)
#         all_t.append(thermo)
#         all_s.append(spectral.repeat(len(belief)))

#     return (
#         torch.cat(all_b),
#         torch.cat(all_s),
#         torch.cat(all_t),
#     )


In [ ]:
# AFRICA_ADAPTER_PATH = "/latest_african_cultural_model/adapter"
# LATAM_ADAPTER_PATH  = "/latest_latin_cultural_model/adapter"

In [ ]:
# from transformers import AutoModelForCausalLM, AutoTokenizer
# from peft import PeftModel

# BASE_MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

# tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
# tokenizer.pad_token = tokenizer.eos_token

# # FP32 base (geometry-safe)
# base_fp32 = AutoModelForCausalLM.from_pretrained(
#     BASE_MODEL_ID,
#     torch_dtype=torch.float32,
#     output_hidden_states=True,
#     device_map="auto"
# ).eval()

# africa_model = PeftModel.from_pretrained(
#     base_fp32,
#     AFRICA_ADAPTER_PATH,
#     is_trainable=False
# ).eval()

# latam_model = PeftModel.from_pretrained(
#     base_fp32,
#     LATAM_ADAPTER_PATH,
#     is_trainable=False
# ).eval()


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
def fisher_style_merge_lora(
    base_model,
    adapter_path_a,
    adapter_path_b,
    alpha: float = 0.5,
):
    """
    Geometry-aware (delta-space) merge of two LoRA adapters.
    alpha ∈ [0,1] controls cultural dominance.
    """

    mA = PeftModel.from_pretrained(
        base_model, adapter_path_a, is_trainable=False
    )
    mB = PeftModel.from_pretrained(
        base_model, adapter_path_b, is_trainable=False
    )

    sdA = mA.state_dict()
    sdB = mB.state_dict()

    merged_sd = {}
    for k in sdA:
        if k in sdB:
            merged_sd[k] = alpha * sdA[k] + (1 - alpha) * sdB[k]
        else:
            merged_sd[k] = sdA[k]

    offspring = PeftModel.from_pretrained(
        base_model, adapter_path_a, is_trainable=False
    )
    offspring.load_state_dict(merged_sd, strict=False)
    return offspring.eval()


In [ ]:
# offspring_model = fisher_style_merge_lora(
#     base_fp32,
#     AFRICA_ADAPTER_PATH,
#     LATAM_ADAPTER_PATH,
#     alpha=0.5
# )

OutOfMemoryError: CUDA out of memory. Tried to allocate 2.00 MiB. GPU 0 has a total capacity of 47.37 GiB of which 1.69 MiB is free. Process 328458 has 47.36 GiB memory in use. Of the allocated memory 42.15 GiB is allocated by PyTorch, and 4.81 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# # for nDNA analysis 64 prompts
# '''
# These probes are never used for training. Only used for geometry.
# '''

# AFRICA_PROBES = [
#     "Explain clan and kinship systems in traditional African communities.",
#     "Describe the importance of elders in African social structures.",
#     "Explain African concepts of community and collective identity.",
#     "Describe traditional African belief systems and spirituality.",
#     "Explain the role of ancestors in African cultural traditions.",
#     "Describe initiation and coming-of-age rituals in Africa.",
#     "Explain the role of music and rhythm in African daily life.",
#     "Describe the cultural significance of drums in Africa.",

#     "Describe marriage and family structures in African societies.",
#     "Explain the role of proverbs in African oral traditions.",
#     "Describe traditional leadership and chieftaincy systems.",
#     "Explain the importance of land and ancestry in African culture.",
#     "Describe African concepts of time and continuity.",
#     "Explain how history is preserved in African oral traditions.",
#     "Describe African approaches to education and learning.",
#     "Explain the role of storytelling in African moral education.",
#     "Describe traditional African festivals and ceremonies.",

#     "Explain the cultural meaning of masks in African societies.",
#     "Describe African artistic traditions and symbolism.",
#     "Explain the role of dance in African cultural expression.",
#     "Describe the social function of African music.",
#     "Explain the cultural importance of communal labor in Africa.",
#     "Describe African hospitality and social etiquette.",
#     "Explain the role of spirituality in everyday African life.",
#     "Describe traditional African healing practices.",
#     "Explain how myths function in African cultures.",
#     "Describe the role of griots in West African societies.",

#     "Explain the significance of lineage in African identity.",
#     "Describe African perspectives on individuality and community.",
#     "Explain how cultural values are transmitted across generations.",
#     "Describe African views on nature and the environment.",
#     "Explain the role of rituals in maintaining social harmony.",
#     "Describe traditional African approaches to justice.",
#     "Explain the cultural meaning of names in African societies.",
#     "Describe the symbolism of animals in African folklore.",
#     "Explain African perspectives on life cycles and death.",
#     "Describe traditional African wedding customs.",

#     "Explain how African societies understand social responsibility.",
#     "Describe the role of respect and hierarchy in African culture.",
#     "Explain African communal decision-making processes.",
#     "Describe traditional African food-sharing practices.",
#     "Explain how African cultures define personal identity.",
#     "Describe the importance of community memory in Africa.",
#     "Explain how African cultures view knowledge and wisdom.",
#     "Describe traditional African rites of passage.",
#     "Explain African perspectives on harmony and balance.",
#     "Describe the cultural role of storytelling during gatherings.",

#     "Explain how African traditions adapt to modern life.",
#     "Describe continuity between ancient and modern African cultures.",
#     "Explain African approaches to resilience and survival.",
#     "Describe how cultural values guide African social behavior."
# ]

# LATAM_PROBES = [
#     "Describe the cultural significance of the Day of the Dead in Latin America.",
#     "Explain indigenous traditions in Latin American societies.",
#     "Describe the role of family in Latin American culture.",
#     "Explain community and social relationships in Latin America.",
#     "Describe traditional celebrations in Latin American countries.",
#     "Explain the influence of indigenous cultures on Latin America.",
#     "Describe cultural identity in Latin American societies.",
#     "Explain the role of religion in Latin American daily life.",
#     "Describe Latin American approaches to community solidarity.",
#     "Explain the importance of festivals in Latin American culture.",

#     "Describe musical traditions across Latin America.",
#     "Explain the cultural role of dance in Latin American societies.",
#     "Describe traditional Latin American artistic expressions.",
#     "Explain how history shapes Latin American cultural identity.",
#     "Describe oral and written storytelling traditions in Latin America.",
#     "Explain the influence of colonial history on Latin American culture.",
#     "Describe traditional family roles in Latin America.",
#     "Explain Latin American views on community responsibility.",
#     "Describe the cultural importance of food in Latin America.",
#     "Explain how cultural values are passed between generations.",

#     "Describe indigenous languages and their cultural significance in Latin America.",
#     "Explain the concept of mestizaje in Latin American societies.",
#     "Describe Afro-Latin cultural influences in Latin America.",
#     "Explain cultural diversity within Latin American countries.",
#     "Describe the role of art in expressing Latin American identity.",
#     "Explain the cultural significance of murals in Latin America.",
#     "Describe Latin American literary traditions.",
#     "Explain the importance of magical realism in Latin American literature.",
#     "Describe storytelling themes common in Latin American culture.",
#     "Explain how cultural memory is preserved in Latin America.",

#     "Describe Latin American perspectives on nature and land.",
#     "Explain the relationship between culture and geography in Latin America.",
#     "Describe rural and urban cultural differences in Latin America.",
#     "Explain traditional healing and folk medicine in Latin America.",
#     "Describe cultural rituals associated with life events in Latin America.",
#     "Explain how Latin American cultures approach death and remembrance.",
#     "Describe the role of music in Latin American social life.",
#     "Explain how dance expresses cultural identity in Latin America.",
#     "Describe the importance of community gatherings in Latin America.",
#     "Explain cultural symbolism in Latin American art.",

#     "Describe how Latin American traditions adapt to modern society.",
#     "Explain cultural continuity across generations in Latin America.",
#     "Describe Latin American approaches to resilience and social change.",
#     "Explain how collective identity is formed in Latin America.",
#     "Describe the influence of migration on Latin American culture.",
#     "Explain cultural expressions of joy and celebration in Latin America.",
#     "Describe the role of storytelling in shaping Latin American values.",
#     "Explain how traditions maintain social cohesion in Latin America.",
#     "Describe Latin American perspectives on cultural heritage.",
#     "Explain how culture shapes everyday behavior in Latin America.",

#     "Describe the relationship between tradition and modernity in Latin America.",
#     "Explain how cultural practices reflect shared values in Latin America.",
#     "Describe how identity is expressed in Latin American communities.",
#     "Explain the role of memory and history in Latin American culture."
# ]

In [ ]:
# b_af, s_af, t_af = ndna_method5_trajectory(
#     africa_model, tokenizer, AFRICA_PROBES
# )

# b_la, s_la, t_la = ndna_method5_trajectory(
#     latam_model, tokenizer, LATAM_PROBES
# )

# b_of, s_of, t_of = ndna_method5_trajectory(
#     offspring_model, tokenizer, AFRICA_PROBES + LATAM_PROBES
# )


AcceleratorError: CUDA error: out of memory
Search for `cudaErrorMemoryAllocation' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# b_af, s_af, t_af

NameError: name 'b_af' is not defined

In [ ]:
# import plotly.graph_objects as go

# def plot_ndna(b, s, t, title):
#     fig = go.Figure(
#         go.Scatter3d(
#             x=b.cpu(),
#             y=s.cpu(),
#             z=t.cpu(),
#             mode="lines",
#             line=dict(width=4)
#         )
#     )
#     fig.update_layout(
#         title=title,
#         scene=dict(
#             xaxis_title="Belief β",
#             yaxis_title="Spectral κ",
#             zaxis_title="Thermodynamic Length ℒ"
#         )
#     )
#     fig.show()

# plot_ndna(b_af, s_af, t_af, "African Model — Method‑5 nDNA")
# plot_ndna(b_la, s_la, t_la, "Latin American Model — Method‑5 nDNA")
# plot_ndna(b_of, s_of, t_of, "Offspring Model — Method‑5 nDNA")


NameError: name 'b_af' is not defined

In [ ]:
# from transformers import AutoModelForCausalLM, AutoTokenizer
# from peft import PeftModel
# import torch

# BASE_MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

# tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
# tokenizer.pad_token = tokenizer.eos_token

# fp32_base = AutoModelForCausalLM.from_pretrained(
#     BASE_MODEL_ID,
#     torch_dtype=torch.float32,
#     output_hidden_states=True,
#     device_map="auto"
# ).eval()


The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
# AFRICA_ADAPTER_PATH = "/latest_african_cultural_model/adapter"
# LATAM_ADAPTER_PATH  = "/latest_latin_cultural_model/adapter"

# africa_model = PeftModel.from_pretrained(
#     fp32_base, AFRICA_ADAPTER_PATH
# ).eval()

# latam_model = PeftModel.from_pretrained(
#     fp32_base, LATAM_ADAPTER_PATH
# ).eval()

# print("✅ African and Latin American models loaded (FP32 + LoRA)")

/venv/main/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:282: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


✅ African and Latin American models loaded (FP32 + LoRA)


In [ ]:
# base_model = AutoModelForCausalLM.from_pretrained(
#     BASE_MODEL_ID,
#     torch_dtype=torch.float32,
#     output_hidden_states=True,
#     device_map="auto"
# ).eval()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
# africa = PeftModel.from_pretrained(
#     base_model,
#     AFRICA_ADAPTER_PATH,
#     is_trainable=False
# )

# latam = PeftModel.from_pretrained(
#     base_model,
#     LATAM_ADAPTER_PATH,
#     is_trainable=False
# )

In [ ]:
# def merge_lora_adapters(
#     base_model,
#     AFRICA_ADAPTER_PATH,
#     LATAM_ADAPTER_PATH,
#     alpha=0.5,
#     beta=0.5
# ):
#     m1 = PeftModel.from_pretrained(base_model, AFRICA_ADAPTER_PATH, is_trainable=False)
#     m2 = PeftModel.from_pretrained(base_model, LATAM_ADAPTER_PATH, is_trainable=False)

#     sd1 = m1.state_dict()
#     sd2 = m2.state_dict()

#     merged_sd = {}
#     for k in sd1:
#         merged_sd[k] = alpha * sd1[k] + beta * sd2.get(k, 0)

#     offspring = PeftModel.from_pretrained(base_model, AFRICA_ADAPTER_PATH, is_trainable=False)
#     offspring.load_state_dict(merged_sd, strict=False)
#     return offspring.eval()


In [ ]:
# import plotly.graph_objects as go

# def plot_ndna(b,s,t,title):
#     fig = go.Figure(go.Scatter3d(
#         x=b.cpu(), y=s.cpu(), z=t.cpu(),
#         mode="lines", line=dict(width=4)
#     ))
#     fig.update_layout(
#         title=title,
#         scene=dict(
#             xaxis_title="Belief β",
#             yaxis_title="Spectral κ",
#             zaxis_title="Thermo Δ"
#         )
#     )
#     fig.show()


In [ ]:
b,s,t = ndna_trajectory(africa_model, AFRICA_PROBES)
plot_ndna(b,s,t,"African Cultural Geometry")

b,s,t = ndna_trajectory(latam_model, LATAM_PROBES)
plot_ndna(b,s,t,"Latin American Cultural Geometry")

b,s,t = ndna_trajectory(offspring_model, AFRICA_PROBES+LATAM_PROBES)
plot_ndna(b,s,t,"Merged Offspring Geometry")

TypeError: 'NoneType' object is not subscriptable

In [ ]:
# @dataclass
# class CulturalConfig:
#     base_model_id: str = "meta-llama/Llama-3.2-3B-Instruct"
#     max_seq_length: int = 512

#     num_train: int = 100
#     num_probe: int = 64   # Method‑5 probes

#     epochs: int = 3
#     batch_size: int = 4
#     grad_accum: int = 4
#     lr: float = 2e-4

#     lora_r: int = 64
#     lora_alpha: int = 128
#     lora_dropout: float = 0.05

#     output_root: str = "/20Dec2025_nDNA_Models/"

In [ ]:
# # for nDNA analysis 64 prompts
# '''
# These probes are never used for training. Only used for geometry.
# '''

# AFRICA_PROBES = [
#     "Explain the Ubuntu philosophy in African societies.",
#     "Describe the role of oral storytelling in African culture.",
#     "Explain clan and kinship systems in traditional African communities.",
#     "Describe the importance of elders in African social structures.",
#     "Explain African concepts of community and collective identity.",
#     "Describe traditional African belief systems and spirituality.",
#     "Explain the role of ancestors in African cultural traditions.",
#     "Describe initiation and coming-of-age rituals in Africa.",
#     "Explain the role of music and rhythm in African daily life.",
#     "Describe the cultural significance of drums in Africa.",

#     "Explain African approaches to conflict resolution.",
#     "Describe marriage and family structures in African societies.",
#     "Explain the role of proverbs in African oral traditions.",
#     "Describe traditional leadership and chieftaincy systems.",
#     "Explain the importance of land and ancestry in African culture.",
#     "Describe African concepts of time and continuity.",
#     "Explain how history is preserved in African oral traditions.",
#     "Describe African approaches to education and learning.",
#     "Explain the role of storytelling in African moral education.",
#     "Describe traditional African festivals and ceremonies.",

#     "Explain the cultural meaning of masks in African societies.",
#     "Describe African artistic traditions and symbolism.",
#     "Explain the role of dance in African cultural expression.",
#     "Describe the social function of African music.",
#     "Explain the cultural importance of communal labor in Africa.",
#     "Describe African hospitality and social etiquette.",
#     "Explain the role of spirituality in everyday African life.",
#     "Describe traditional African healing practices.",
#     "Explain how myths function in African cultures.",
#     "Describe the role of griots in West African societies.",

#     "Explain the significance of lineage in African identity.",
#     "Describe African perspectives on individuality and community.",
#     "Explain how cultural values are transmitted across generations.",
#     "Describe African views on nature and the environment.",
#     "Explain the role of rituals in maintaining social harmony.",
#     "Describe traditional African approaches to justice.",
#     "Explain the cultural meaning of names in African societies.",
#     "Describe the symbolism of animals in African folklore.",
#     "Explain African perspectives on life cycles and death.",
#     "Describe traditional African wedding customs.",

#     "Explain how African societies understand social responsibility.",
#     "Describe the role of respect and hierarchy in African culture.",
#     "Explain African communal decision-making processes.",
#     "Describe traditional African food-sharing practices.",
#     "Explain how African cultures define personal identity.",
#     "Describe the importance of community memory in Africa.",
#     "Explain how African cultures view knowledge and wisdom.",
#     "Describe traditional African rites of passage.",
#     "Explain African perspectives on harmony and balance.",
#     "Describe the cultural role of storytelling during gatherings.",

#     "Explain how African traditions adapt to modern life.",
#     "Describe continuity between ancient and modern African cultures.",
#     "Explain African approaches to resilience and survival.",
#     "Describe how cultural values guide African social behavior."
# ]

# LATAM_PROBES = [
#     "Describe the cultural significance of the Day of the Dead in Latin America.",
#     "Explain indigenous traditions in Latin American societies.",
#     "Describe the role of family in Latin American culture.",
#     "Explain community and social relationships in Latin America.",
#     "Describe traditional celebrations in Latin American countries.",
#     "Explain the influence of indigenous cultures on Latin America.",
#     "Describe cultural identity in Latin American societies.",
#     "Explain the role of religion in Latin American daily life.",
#     "Describe Latin American approaches to community solidarity.",
#     "Explain the importance of festivals in Latin American culture.",

#     "Describe musical traditions across Latin America.",
#     "Explain the cultural role of dance in Latin American societies.",
#     "Describe traditional Latin American artistic expressions.",
#     "Explain how history shapes Latin American cultural identity.",
#     "Describe oral and written storytelling traditions in Latin America.",
#     "Explain the influence of colonial history on Latin American culture.",
#     "Describe traditional family roles in Latin America.",
#     "Explain Latin American views on community responsibility.",
#     "Describe the cultural importance of food in Latin America.",
#     "Explain how cultural values are passed between generations.",

#     "Describe indigenous languages and their cultural significance in Latin America.",
#     "Explain the concept of mestizaje in Latin American societies.",
#     "Describe Afro-Latin cultural influences in Latin America.",
#     "Explain cultural diversity within Latin American countries.",
#     "Describe the role of art in expressing Latin American identity.",
#     "Explain the cultural significance of murals in Latin America.",
#     "Describe Latin American literary traditions.",
#     "Explain the importance of magical realism in Latin American literature.",
#     "Describe storytelling themes common in Latin American culture.",
#     "Explain how cultural memory is preserved in Latin America.",

#     "Describe Latin American perspectives on nature and land.",
#     "Explain the relationship between culture and geography in Latin America.",
#     "Describe rural and urban cultural differences in Latin America.",
#     "Explain traditional healing and folk medicine in Latin America.",
#     "Describe cultural rituals associated with life events in Latin America.",
#     "Explain how Latin American cultures approach death and remembrance.",
#     "Describe the role of music in Latin American social life.",
#     "Explain how dance expresses cultural identity in Latin America.",
#     "Describe the importance of community gatherings in Latin America.",
#     "Explain cultural symbolism in Latin American art.",

#     "Describe how Latin American traditions adapt to modern society.",
#     "Explain cultural continuity across generations in Latin America.",
#     "Describe Latin American approaches to resilience and social change.",
#     "Explain how collective identity is formed in Latin America.",
#     "Describe the influence of migration on Latin American culture.",
#     "Explain cultural expressions of joy and celebration in Latin America.",
#     "Describe the role of storytelling in shaping Latin American values.",
#     "Explain how traditions maintain social cohesion in Latin America.",
#     "Describe Latin American perspectives on cultural heritage.",
#     "Explain how culture shapes everyday behavior in Latin America.",

#     "Describe the relationship between tradition and modernity in Latin America.",
#     "Explain how cultural practices reflect shared values in Latin America.",
#     "Describe how identity is expressed in Latin American communities.",
#     "Explain the role of memory and history in Latin American culture."
# ]

In [ ]:
# # ============================================================================
# # CELL 7: LOAD BASE MODEL AND TOKENIZER
# # ============================================================================

# print("\n📥 Loading base model and tokenizer...")

# # Quantization config for memory efficiency
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=COMPUTE_DTYPE,
#     bnb_4bit_use_double_quant=True,
# )

# # Load tokenizer
# tokenizer = AutoTokenizer.from_pretrained(
#     CulturalConfig.base_model_id,
#     trust_remote_code=True
# )
# tokenizer.pad_token = tokenizer.eos_token
# tokenizer.padding_side = "right"

# print(f"   ✅ Tokenizer loaded: vocab size = {len(tokenizer)}")

# # Load base model
# base_model = AutoModelForCausalLM.from_pretrained(
#     CulturalConfig.base_model_id,
#     quantization_config=bnb_config,
#     device_map="auto",
#     trust_remote_code=True,
#     torch_dtype=COMPUTE_DTYPE,
# )

# print(f"   ✅ Base model loaded")
# print(f"   Model type: {type(base_model).__name__}")
# print(f"   Number of layers: {base_model.config.num_hidden_layers}")

# # Update config with actual layer count
# CulturalConfig.num_layers = base_model.config.num_hidden_layers


📥 Loading base model and tokenizer...


`torch_dtype` is deprecated! Use `dtype` instead!


   ✅ Tokenizer loaded: vocab size = 128256


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

   ✅ Base model loaded
   Model type: LlamaForCausalLM
   Number of layers: 28


In [ ]:
# from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# base_model = prepare_model_for_kbit_training(base_model)

# lora_cfg = LoraConfig(
#     r=CulturalConfig.lora_r,
#     lora_alpha=CulturalConfig.lora_alpha,
#     target_modules=["q_proj","k_proj","v_proj","o_proj",
#                     "gate_proj","up_proj","down_proj"],
#     lora_dropout=CulturalConfig.lora_dropout,
#     bias="none",
#     task_type="CAUSAL_LM"
# )

# model = get_peft_model(base_model, lora_cfg)
# model.print_trainable_parameters()


trainable params: 97,255,424 || all params: 3,310,005,248 || trainable%: 2.9382


In [ ]:
# model.save_pretrained("africa_lora")
# tokenizer.save_pretrained("africa_lora")

('africa_lora/tokenizer_config.json',
 'africa_lora/special_tokens_map.json',
 'africa_lora/chat_template.jinja',
 'africa_lora/tokenizer.json')

In [ ]:
# model.save_pretrained("latam_lora")
# tokenizer.save_pretrained("latam_lora")

('latam_lora/tokenizer_config.json',
 'latam_lora/special_tokens_map.json',
 'latam_lora/chat_template.jinja',
 'latam_lora/tokenizer.json')

#TRAIN AFRICA MODEL

In [ ]:
# from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

# args = TrainingArguments(
#     output_dir="./africa_model",
#     num_train_epochs=CulturalConfig.epochs,
#     per_device_train_batch_size=CulturalConfig.batch_size,
#     gradient_accumulation_steps=CulturalConfig.grad_accum,
#     learning_rate=CulturalConfig.lr,
#     logging_steps=50,
#     save_steps=500,
#     fp16=True,
#     optim="paged_adamw_8bit",
#     report_to="none"
# )

In [ ]:
# adapter_path = os.path.join("africa_lora", "adapter")
# model.save_pretrained(adapter_path)
# tokenizer.save_pretrained(adapter_path)

('africa_lora/adapter/tokenizer_config.json',
 'africa_lora/adapter/special_tokens_map.json',
 'africa_lora/adapter/chat_template.jinja',
 'africa_lora/adapter/tokenizer.json')

In [ ]:
# adapter_path = os.path.join("latam_lora", "adapter")
# model.save_pretrained(adapter_path)
# tokenizer.save_pretrained(adapter_path)

('latam_lora/adapter/tokenizer_config.json',
 'latam_lora/adapter/special_tokens_map.json',
 'latam_lora/adapter/chat_template.jinja',
 'latam_lora/adapter/tokenizer.json')

In [ ]:
# from peft import PeftModel

# fp32_base = AutoModelForCausalLM.from_pretrained(
#     CulturalConfig.base_model_id,
#     torch_dtype=torch.float32,
#     output_hidden_states=True,
#     device_map="auto"
# )

# AFRICA_ADAPTER_PATH = (
#     "/africa_lora/adapter"
# )

# LATAM_ADAPTER_PATH = (
#     "/latam_lora/adapter"
# )

# africa_model = PeftModel.from_pretrained(
#     fp32_base,
#     AFRICA_ADAPTER_PATH,
# ).eval()


# latam_model = PeftModel.from_pretrained(
#     fp32_base,
#     LATAM_ADAPTER_PATH
# ).eval()


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 2.00 MiB. GPU 0 has a total capacity of 44.42 GiB of which 2.06 MiB is free. Process 1148173 has 15.92 GiB memory in use. Process 1224188 has 28.49 GiB memory in use. Of the allocated memory 27.84 GiB is allocated by PyTorch, and 170.93 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)